# AR-CBT V11 — Vérification hors GPU

Ce notebook vérifie les fichiers existants, sans appeler de modèle ni produire de nouveaux scores. Ajoutez le dossier public V11 comme entrée Kaggle. Les données source et les résultats primaires restent inchangés. Les diagnostics supplémentaires sont post-dévoilement, non une validation clinique.


In [ ]:
from pathlib import Path
import shutil, sys, subprocess
from datetime import datetime, timezone
# Rechercher uniquement le package V11, sans supposer un compte Kaggle.
candidates = list(Path('/kaggle/input').rglob('prospective_pilot/code/reproduce_pilot.py')) if Path('/kaggle/input').exists() else list(Path.cwd().rglob('prospective_pilot/code/reproduce_pilot.py'))
roots = sorted({p.parents[2] for p in candidates})
if len(roots) != 1:
    raise RuntimeError(f'Un seul package V11 doit être présent. Racines trouvées : {roots}')
source = roots[0]
base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
work = base / ('ARCBT_V11_CHECK_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
shutil.copytree(source, work)
print('Copie de travail :', work)


In [ ]:
subprocess.run([sys.executable, str(work/'prospective_pilot/code/reproduce_pilot.py')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(work/'prospective_pilot/tests'), '-v'], check=True)
print('Contrôles hors GPU terminés. Aucun LLM appelé.')


In [ ]:
import zipfile, os
from IPython.display import FileLink, display
output = work/'prospective_pilot/analysis_v11'
archive = base/(work.name + '_RESULTS.zip')
with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(output.rglob('*')):
        if path.is_file(): z.write(path, path.relative_to(output))
print(archive)
display(FileLink(os.path.relpath(archive)))


## Appels GPU : non lancés par ce notebook

Les programmes `replay_models.py` et `generate_new_dialogues.py` produisent uniquement de nouvelles exécutions dans un dossier séparé. Leur exécution GPU n'a pas été réalisée dans l'audit V11. Ne remplacez jamais les sorties originales par une exécution nouvelle et ne présentez pas une réexécution comme validation humaine ou test du Guardian.
